In [23]:
from neo4j import GraphDatabase
import pandas as pd

In [24]:
df = pd.read_csv('fpl_m3.csv')

In [25]:
df = df.rename(columns={'season_x': 'season'})

In [26]:
df['transfers_balance'] = df['transfers_in'] - df['transfers_out']

In [27]:
df['home_team'] = df.apply(lambda x: x['team_x'] if x['was_home'] else x['opp_team_name'], axis=1)
df['away_team'] = df.apply(lambda x: x['opp_team_name'] if x['was_home'] else x['team_x'], axis=1)

In [28]:
df['kickoff_time'] = pd.to_datetime(df['kickoff_time']).dt.strftime('%Y-%m-%d %H:%M:%S+00:00')

In [29]:
df = df.drop(columns=['was_home', 'G_A', 'pos_DEF', 'pos_FWD', 'pos_GK', 'pos_MID', 'team_x_global_code', 'opponent_team_global_code', 
                      'upcoming_total_points', 'season_x_code', 'team_strength', 'opponent_strength', 'strength_difference', 'round', 'opponent_team'])

In [30]:
df.to_csv('fpl_graph_final.csv', index=False)

In [31]:
config = {}
with open("config.txt", "r") as f:
    for line in f:
        key, value = line.strip().split("=", 1)
        config[key] = value

driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))

df = pd.read_csv("fpl_graph_final.csv", header=0)

with driver.session() as session:
    batch_size = 1000

    data = []
    for row in df.itertuples():
        data.append({
            'season': row.season,
            'GW': row.GW,
            'fixture': row.fixture,
            'home_team': row.home_team,
            'away_team': row.away_team,
            'team_x': row.team_x,               # ADDED
            'opp_team_name': row.opp_team_name, # ADDED
            'name': row.name,
            'element': row.element,
            'code': row.code,                   # ADDED
            'position': row.position,
            'kickoff': row.kickoff_time,
            'minutes': row.minutes,
            'goals_scored': row.goals_scored,
            'assists': row.assists,
            'total_points': row.total_points,
            'bonus': row.bonus,
            'clean_sheets': row.clean_sheets,
            'goals_conceded': row.goals_conceded,
            'own_goals': row.own_goals,
            'penalties_saved': row.penalties_saved,
            'penalties_missed': row.penalties_missed,
            'yellow_cards': row.yellow_cards,
            'red_cards': row.red_cards,
            'saves': row.saves,
            'bps': row.bps,
            'influence': row.influence,
            'creativity': row.creativity,
            'threat': row.threat,
            'ict_index': row.ict_index,
            'form': row.form
        })

    for i in range(0, len(data), batch_size):
        batch = data[i:i + batch_size]

        session.run(
            """
            UNWIND $batch as row

            // === NODES ===
            MERGE (s:Season {season_name: row.season})
            MERGE (g:Gameweek {season: row.season, GW_number: row.GW})
            MERGE (f:Fixture {season: row.season, fixture_number: row.fixture})
                SET f.kickoff_time = row.kickoff

            // Teams
            MERGE (t_home:Team {name: row.home_team})
            MERGE (t_away:Team {name: row.away_team})
            MERGE (t_player:Team {name: row.team_x})             // ADDED
            MERGE (t_opp:Team {name: row.opp_team_name})         // ADDED

            // Player
            MERGE (p:Player {
                player_name: row.name,
                player_element: row.element
            })
            SET p.code = row.code                                // ADDED

            // Position
            MERGE (pos:Position {name: row.position})


            // === RELATIONSHIPS ===

            // Season → GW
            MERGE (s)-[:HAS_GW]->(g)

            // GW → Fixture
            MERGE (g)-[:HAS_FIXTURE]->(f)

            // Fixture teams
            MERGE (f)-[:HAS_HOME_TEAM]->(t_home)
            MERGE (f)-[:HAS_AWAY_TEAM]->(t_away)

            // Player → Position
            MERGE (p)-[:PLAYS_AS]->(pos)

            // Player → Team (career/season relationship)
            MERGE (p)-[:PLAYS_FOR {season: row.season}]->(t_player)   // ADDED

            // Opponent link (useful for queries)
            MERGE (p)-[:PLAYED_AGAINST]->(t_opp)                       // ADDED

            // Player → Fixture performance stats
            MERGE (p)-[r:PLAYED_IN]->(f)
            SET
                r.minutes = row.minutes,
                r.goals_scored = row.goals_scored,
                r.assists = row.assists,
                r.total_points = row.total_points,
                r.bonus = row.bonus,
                r.clean_sheets = row.clean_sheets,
                r.goals_conceded = row.goals_conceded,
                r.own_goals = row.own_goals,
                r.penalties_saved = row.penalties_saved,
                r.penalties_missed = row.penalties_missed,
                r.yellow_cards = row.yellow_cards,
                r.red_cards = row.red_cards,
                r.saves = row.saves,
                r.bps = row.bps,
                r.influence = row.influence,
                r.creativity = row.creativity,
                r.threat = row.threat,
                r.ict_index = row.ict_index,
                r.form = row.form
            """,
            batch=batch
        )

        print(f"Imported batch {i // batch_size + 1}: rows {i} to {min(i + batch_size, len(data))}")

    print("Data imported successfully.")

driver.close()


Imported batch 1: rows 0 to 1000
Imported batch 2: rows 1000 to 2000
Imported batch 3: rows 2000 to 3000
Imported batch 4: rows 3000 to 4000
Imported batch 5: rows 4000 to 5000
Imported batch 6: rows 5000 to 6000
Imported batch 7: rows 6000 to 7000
Imported batch 8: rows 7000 to 8000
Imported batch 9: rows 8000 to 9000
Imported batch 10: rows 9000 to 10000
Imported batch 11: rows 10000 to 11000
Imported batch 12: rows 11000 to 12000
Imported batch 13: rows 12000 to 13000
Imported batch 14: rows 13000 to 14000
Imported batch 15: rows 14000 to 15000
Imported batch 16: rows 15000 to 16000
Imported batch 17: rows 16000 to 17000
Imported batch 18: rows 17000 to 18000
Imported batch 19: rows 18000 to 19000
Imported batch 20: rows 19000 to 20000
Imported batch 21: rows 20000 to 21000
Imported batch 22: rows 21000 to 22000
Imported batch 23: rows 22000 to 23000
Imported batch 24: rows 23000 to 24000
Imported batch 25: rows 24000 to 25000
Imported batch 26: rows 25000 to 26000
Imported batch 27